<a href="https://colab.research.google.com/github/Lstemm/LRS.DSC-525/blob/main/LRS.DSC-525.Lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/databricks/LearningSparkV2/refs/heads/master/databricks-datasets/learning-spark-v2/flights/departuredelays.csv

--2026-07-04 04:19:34--  https://raw.githubusercontent.com/databricks/LearningSparkV2/refs/heads/master/databricks-datasets/learning-spark-v2/flights/departuredelays.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 33396236 (32M) [text/plain]
Saving to: ‘departuredelays.csv’

departuredelays.csv 100%[===================>]  31.85M   179MB/s    in 0.2s    

2026-07-04 04:19:35 (179 MB/s) - ‘departuredelays.csv’ saved [33396236/33396236]



In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Example").enableHiveSupport(). getOrCreate()
sc = spark.sparkContext #for backward compatibility, RDD would need this

In [3]:
#to be on the safe side I prepare the schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("date", StringType(), True),        # date as string!!!
    StructField("delay", IntegerType(), True),
    StructField("distance", IntegerType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True)
])


In [4]:
df = spark.read.option("header", "true").schema(schema).csv("departuredelays.csv")
#I prepared schema to make sure date column is read as string type.

In [5]:
df.printSchema()
df.show(3)

root
 |-- date: string (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows


In [6]:
from pyspark.sql.functions import col, lpad, substring, concat_ws, to_date,to_timestamp, current_date, concat, year

# Pad to 7 digits: e.g., 1010005 -> "1010005"
df = df.withColumn("date_str", lpad(col("date").cast("string"), 7, "0")) #add zero to the left
df = df.withColumn("as_time_stamp", to_timestamp(concat(year(current_date()), col("date")), "yyyyMMddHHmm") )








In [7]:
df.printSchema()
df.show(3)

root
 |-- date: string (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- date_str: string (nullable = true)
 |-- as_time_stamp: timestamp (nullable = true)

+--------+-----+--------+------+-----------+--------+-------------------+
|    date|delay|distance|origin|destination|date_str|      as_time_stamp|
+--------+-----+--------+------+-----------+--------+-------------------+
|01011245|    6|     602|   ABE|        ATL| 0101124|2026-01-01 12:45:00|
|01020600|   -8|     369|   ABE|        DTW| 0102060|2026-01-02 06:00:00|
|01021245|   -2|     602|   ABE|        ATL| 0102124|2026-01-02 12:45:00|
+--------+-----+--------+------+-----------+--------+-------------------+
only showing top 3 rows


In [8]:
df.createOrReplaceTempView("us_delay_flights_tbl")

In [9]:
spark.sql("CREATE DATABASE my_spark_db")
spark.sql("USE my_spark_db")


DataFrame[]

In [10]:
spark.sql("CREATE TABLE dummy(date STRING, delay INT,distance INT, origin STRING, destination STRING)") #we may need HIVE setup for this

DataFrame[]

In [11]:
df.write.saveAsTable("managed_us_delay_flights_tbl2") #managed


In [12]:
df.write.option("path", "/tmp/data/us_flights_delay").saveAsTable("us_delay_flights_tbl_unmanaged") #unmanaged

In [14]:
# Create another session
spark2 = spark.newSession()

# Create temp view in first session
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df.createOrReplaceTempView("people")

# Query in first session -> works
spark.sql("SELECT * FROM us_delay_flights_tbl_unmanaged where delay<0").show()

# Query in second session -> fails (temp view not shared)
#spark2.sql("SELECT * FROM us_delay_flights_tbl_unmanaged where delay<0").show() # Would error

# Each session has its own catalog of temp views
df2 = spark2.createDataFrame([(3, "Carol")], ["id", "name"])
df2.createOrReplaceTempView("people2")

# Accessible in spark2 but not in spark
spark2.sql("SELECT * FROM people2").show()

+--------+-----+--------+------+-----------+--------+-------------------+
|    date|delay|distance|origin|destination|date_str|      as_time_stamp|
+--------+-----+--------+------+-----------+--------+-------------------+
|01020600|   -8|     369|   ABE|        DTW| 0102060|2026-01-02 06:00:00|
|01021245|   -2|     602|   ABE|        ATL| 0102124|2026-01-02 12:45:00|
|01020605|   -4|     602|   ABE|        ATL| 0102060|2026-01-02 06:05:00|
|01031245|   -4|     602|   ABE|        ATL| 0103124|2026-01-03 12:45:00|
|01061215|   -6|     602|   ABE|        ATL| 0106121|2026-01-06 12:15:00|
|01060625|   -3|     602|   ABE|        ATL| 0106062|2026-01-06 06:25:00|
|01091230|   -4|     369|   ABE|        DTW| 0109123|2026-01-09 12:30:00|
|01101215|   -5|     602|   ABE|        ATL| 0110121|2026-01-10 12:15:00|
|01100600|   -5|     369|   ABE|        DTW| 0110060|2026-01-10 06:00:00|
|01101230|   -8|     369|   ABE|        DTW| 0110123|2026-01-10 12:30:00|
|01110600|   -9|     369|   ABE|      

In [15]:
print(spark.catalog.listDatabases())
print(spark.catalog.listTables())
spark.catalog.listColumns("us_delay_flights_tbl")

[Database(name='default', catalog='spark_catalog', description='Default Hive database', locationUri='file:/content/spark-warehouse'), Database(name='my_spark_db', catalog='spark_catalog', description='', locationUri='file:/content/spark-warehouse/my_spark_db.db')]
[Table(name='dummy', catalog='spark_catalog', namespace=['my_spark_db'], description=None, tableType='MANAGED', isTemporary=False), Table(name='managed_us_delay_flights_tbl2', catalog='spark_catalog', namespace=['my_spark_db'], description=None, tableType='MANAGED', isTemporary=False), Table(name='us_delay_flights_tbl_unmanaged', catalog='spark_catalog', namespace=['my_spark_db'], description=None, tableType='EXTERNAL', isTemporary=False), Table(name='people', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='us_delay_flights_tbl', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


[Column(name='date', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='delay', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='distance', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='origin', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='destination', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='date_str', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='as_time_stamp', description=None, dataType='timestamp', nullable=True, isPartition=False, isBucket=False, isCluster=False)]

Can you get the m&m data from https://raw.githubusercontent.com/databricks/LearningSparkV2/refs/heads/master/chapter2/py/src/data/mnm_dataset.csv and perform the following:

1. Read it as a csv file
2. Register as a view
3. Select from the DataFrame the fields "State", "Color", and "Count" in a way that it should report the count of each color for each state. In other words, grouping by state and color pairs should be counted.
4. Report what if we just want to see the data for a single state, e.g., CA?


### 1. Read it as a csv file.

In [16]:
# Download the mnm_dataset.csv file
!wget https://raw.githubusercontent.com/databricks/LearningSparkV2/refs/heads/master/chapter2/py/src/data/mnm_dataset.csv

--2026-07-04 04:32:41--  https://raw.githubusercontent.com/databricks/LearningSparkV2/refs/heads/master/chapter2/py/src/data/mnm_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1284872 (1.2M) [text/plain]
Saving to: ‘mnm_dataset.csv’

mnm_dataset.csv     100%[===================>]   1.22M  --.-KB/s    in 0.05s   

2026-07-04 04:32:42 (23.1 MB/s) - ‘mnm_dataset.csv’ saved [1284872/1284872]



2. Register as a view

In [17]:
# Read the mnm_dataset.csv into a Spark DataFrame
mnm_df = spark.read.option("header", "true").option("inferSchema", "true").csv("mnm_dataset.csv")

# Register the DataFrame as a temporary view
mnm_df.createOrReplaceTempView("mnm_view")

print("M&M DataFrame schema:")
mnm_df.printSchema()
print("First 5 rows of M&M DataFrame:")
mnm_df.show(5)

M&M DataFrame schema:
root
 |-- State: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- Count: integer (nullable = true)

First 5 rows of M&M DataFrame:
+-----+------+-----+
|State| Color|Count|
+-----+------+-----+
|   TX|   Red|   20|
|   NV|  Blue|   66|
|   CO|  Blue|   79|
|   OR|  Blue|   71|
|   WA|Yellow|   93|
+-----+------+-----+
only showing top 5 rows


### 3. Select from the DataFrame the fields "State", "Color", and "Count" in a way that it should report the count of each color for each state.

In [18]:
from pyspark.sql.functions import sum, col

# Group by State and Color and count the occurrences
mnm_counts_df = mnm_df.groupBy("State", "Color").agg(sum("Count").alias("Total_Count"))

print("Count of each color for each state:")
mnm_counts_df.show(10)

Count of each color for each state:
+-----+------+-----------+
|State| Color|Total_Count|
+-----+------+-----------+
|   WY| Green|      94339|
|   NV|   Red|      89346|
|   UT|  Blue|      89977|
|   WA|Orange|      91521|
|   NM| Green|      91160|
|   CA|  Blue|      89123|
|   WA|   Red|      93332|
|   NV| Brown|      92478|
|   AZ| Green|      91882|
|   CA|   Red|      91527|
+-----+------+-----------+
only showing top 10 rows


### 4. Report what if we just want to see the data for a single state, e.g., CA?

In [19]:
# Filter for a single state, e.g., 'CA'
ca_mnm_counts_df = mnm_counts_df.filter(col("State") == "CA")

print("Count of each color for CA state:")
ca_mnm_counts_df.show(10)

Count of each color for CA state:
+-----+------+-----------+
|State| Color|Total_Count|
+-----+------+-----------+
|   CA|  Blue|      89123|
|   CA|   Red|      91527|
|   CA|Yellow|     100956|
|   CA|Orange|      90311|
|   CA| Brown|      95762|
|   CA| Green|      93505|
+-----+------+-----------+

